# 04 · Entanglement and Multi-Photon States

**Covers:** tensor products, product vs entangled states, Bell states, building entanglement with Hadamard + CNOT, and the entropy of entanglement.
**Photonics thread:** SPDC crystals and microring resonators produce entangled photon pairs; quantum gates and quantum communication run on them.

In [1]:
import numpy as np
import qutip as qt

results = []   # every theory-vs-simulation check is recorded here

def check(label, sim, theory, tol=1e-3):
    """Compare a simulated number with its analytic value and record the result."""
    sim, theory = float(np.real(sim)), float(np.real(theory))
    ok = bool(np.isclose(sim, theory, atol=tol))
    results.append(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {label}: simulation = {sim:.4f}, theory = {theory:.4f}")

In [2]:
H, V = qt.basis(2, 0), qt.basis(2, 1)
Had = qt.Qobj([[1, 1], [1, -1]]) / np.sqrt(2)
plus = Had * H                                           # 45 degrees

def amps(s):
    return s.full().flatten()

## 1. Tensor product: describing two photons

**Plain words:** one photon has 2 options, two photons have 2 × 2 = 4 joint options (HH, HV, VH, VV). We write one list of 4 amplitudes:

$$(a_0, a_1)\otimes(b_0, b_1) = (a_0b_0,\ a_0b_1,\ a_1b_0,\ a_1b_1)$$

**Gap:** $n$ photons need $2^n$ amplitudes. That exponential growth is why classical computers struggle to simulate quantum systems.

**One sentence:** two photons live in one list of four numbers built by multiplying the single-photon lists.

In [3]:
hv = qt.tensor(H, V)
print("H x V =", amps(hv).real)
check("H x V has amplitude 1 at position HV", amps(hv)[1].real, 1.0)
prod = qt.tensor(plus, plus)
print("45 x 45 =", amps(prod).real.round(3))
check("every entry of 45 x 45 is 0.5", amps(prod)[3].real, 0.5)

H x V = [0. 1. 0. 0.]
PASS  H x V has amplitude 1 at position HV: simulation = 1.0000, theory = 1.0000
45 x 45 = [0.5 0.5 0.5 0.5]
PASS  every entry of 45 x 45 is 0.5: simulation = 0.5000, theory = 0.5000


## 2. Product vs entangled

A **product state** gives each photon its own state. An **entangled state** cannot be split that way. The Bell state

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt2}$$

fails the split: for any product state $c_{00}c_{11} - c_{01}c_{10} = 0$, while Bell gives 0.5.

**What you see:** each photon alone is 50/50 random, yet the two always agree.

**Gap:** this sends no signal faster than light: the agreement only shows when the two lists are compared. Matching H/V results alone could be faked with pre-agreed coins; notebook 05 shows the test the coins fail.

**One sentence:** an entangled pair has no individual states, only a shared one.

In [4]:
bell = (qt.tensor(H, H) + qt.tensor(V, V)).unit()

def concurrence_test(state):
    c = amps(state)
    return abs(c[0] * c[3] - c[1] * c[2])

check("split test, product state", concurrence_test(prod), 0.0)
check("split test, Bell state", concurrence_test(bell), 0.5)
probs = np.abs(amps(bell)) ** 2
print("P(HH, HV, VH, VV) =", probs.round(3))
check("P(H on photon 1, V on photon 2)", probs[1], 0.0)

PASS  split test, product state: simulation = 0.0000, theory = 0.0000
PASS  split test, Bell state: simulation = 0.5000, theory = 0.5000
P(HH, HV, VH, VV) = [0.5 0.  0.  0.5]
PASS  P(H on photon 1, V on photon 2): simulation = 0.0000, theory = 0.0000


## 3. Making entanglement with gates

Hadamard on photon 1, then CNOT:

$$|00\rangle \xrightarrow{H\otimes I} \frac{|00\rangle + |10\rangle}{\sqrt2} \xrightarrow{\text{CNOT}} \frac{|00\rangle + |11\rangle}{\sqrt2}$$

**Photonics:** chips usually get entanglement directly from pair sources (SPDC, four-wave mixing), because a linear-optics CNOT needs extra photons and measurements.

**One sentence:** a superposition plus a conditional flip makes a Bell state.

In [5]:
CNOT = qt.Qobj([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], dims=[[2, 2], [2, 2]])
made = CNOT * qt.tensor(Had, qt.qeye(2)) * qt.tensor(H, H)
check("H + CNOT builds the Bell state", abs(bell.dag() * made) ** 2, 1.0)

PASS  H + CNOT builds the Bell state: simulation = 1.0000, theory = 1.0000


## 4. How much entanglement? Entropy of entanglement

Look at one photon of the pair and ignore the other (the *partial trace*, explained fully in notebook 05). For a product state the single photon is still pure; for the Bell state it is completely random. The von Neumann entropy of that single photon measures the entanglement: 0 bits for a product state, 1 bit (one "ebit") for a Bell state.

**One sentence:** the more random one photon looks on its own, the more entangled the pair.

In [6]:
def ent_entropy(state):
    return qt.entropy_vn(qt.ket2dm(state).ptrace(0), base=2)
check("entropy, product state (bits)", ent_entropy(prod), 0.0)
check("entropy, Bell state (bits)", ent_entropy(bell), 1.0)

# a partially entangled state: cos(t)|00> + sin(t)|11>
t = np.pi / 8
partial = np.cos(t) * qt.tensor(H, H) + np.sin(t) * qt.tensor(V, V)
pc, ps = np.cos(t) ** 2, np.sin(t) ** 2
check("entropy, partial entanglement", ent_entropy(partial), -pc * np.log2(pc) - ps * np.log2(ps))

PASS  entropy, product state (bits): simulation = 0.0000, theory = 0.0000
PASS  entropy, Bell state (bits): simulation = 1.0000, theory = 1.0000
PASS  entropy, partial entanglement: simulation = 0.6009, theory = 0.6009


## Validation summary

Every `check` above compared a QuTiP result with its analytic formula.

In [7]:
print(f"{sum(results)} of {len(results)} checks passed")
assert all(results), "a simulation disagrees with theory"

9 of 9 checks passed
